# Recompra: atraso e avaliação

Como atraso e avaliação da compra de referência se relacionam com a recompra? Mantém a regra atual, que permite recompra anterior à entrega.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RAIZ_PROJETO = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / 'executar_pipeline.py').is_file() and (p / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Abra o notebook dentro da pasta do projeto.')

# Os resumos de outputs/tabelas/parquet não têm o detalhe necessário.
PASTA_BASES = RAIZ_PROJETO / 'data' / 'processed'
df_pedidos = pd.read_parquet(PASTA_BASES / 'base_analitica_pedidos.parquet')
assert df_pedidos['order_id'].notna().all()
assert df_pedidos['order_id'].is_unique
pd.set_option('display.max_columns', None)


In [ ]:
assert df_pedidos['customer_unique_id'].notna().all()
assert pd.api.types.is_datetime64_any_dtype(df_pedidos['order_purchase_timestamp'])
DATA_FIM_OBSERVACAO = df_pedidos['order_purchase_timestamp'].max()
assert pd.notna(DATA_FIM_OBSERVACAO)
MINIMO_ACOMPANHAMENTO = pd.Timedelta(days=90)

# Preserva a ordem das operações do notebook 06, inclusive a exclusão
# de entrega ausente APÓS selecionar a primeira compra entregue.
df_referencia = (
    df_pedidos.loc[df_pedidos['pedido_entregue'].eq(True), [
        'customer_unique_id', 'order_id', 'order_purchase_timestamp',
        'order_delivered_customer_date', 'review_score', 'is_late'
    ]]
    .sort_values(['order_purchase_timestamp', 'order_id'])
    .drop_duplicates('customer_unique_id', keep='first')
    .dropna(subset=['order_delivered_customer_date'])
    .rename(columns={
        'order_id': 'pedido_referencia',
        'order_purchase_timestamp': 'data_compra_referencia',
        'order_delivered_customer_date': 'data_entrega_referencia',
        'review_score': 'avaliacao_referencia',
        'is_late': 'atraso_referencia'
    })
)
df_referencia = df_referencia.loc[
    (df_referencia['data_entrega_referencia'] + MINIMO_ACOMPANHAMENTO)
    .le(DATA_FIM_OBSERVACAO)
].copy()
assert df_referencia['customer_unique_id'].is_unique
assert df_referencia['pedido_referencia'].is_unique
assert df_referencia['avaliacao_referencia'].dropna().isin([1, 2, 3, 4, 5]).all()


In [ ]:
# Cada cliente pode ter vários pedidos candidatos; a referência é única.
df_candidatos = df_pedidos[[
    'customer_unique_id', 'order_id', 'order_purchase_timestamp'
]].merge(
    df_referencia[['customer_unique_id', 'pedido_referencia', 'data_compra_referencia']],
    on='customer_unique_id', how='inner', validate='many_to_one'
)
df_proxima_compra = (
    df_candidatos.loc[
        df_candidatos['order_id'].ne(df_candidatos['pedido_referencia'])
        & df_candidatos['order_purchase_timestamp'].gt(df_candidatos['data_compra_referencia'])
    ]
    .sort_values(['customer_unique_id', 'order_purchase_timestamp', 'order_id'])
    .drop_duplicates('customer_unique_id', keep='first')
    [['customer_unique_id', 'order_id', 'order_purchase_timestamp']]
    .rename(columns={
        'order_id': 'pedido_recompra',
        'order_purchase_timestamp': 'data_recompra'
    })
)
# Left join mantém quem NÃO recomprou no denominador.
df_clientes_recompra = df_referencia.merge(
    df_proxima_compra, on='customer_unique_id', how='left', validate='one_to_one'
).reset_index(drop=True)
df_clientes_recompra['recomprou'] = df_clientes_recompra['pedido_recompra'].notna()
df_clientes_recompra['dias_ate_recompra'] = (
    df_clientes_recompra['data_recompra'] - df_clientes_recompra['data_compra_referencia']
).dt.total_seconds().div(86400)
df_clientes_recompra['dias_observaveis_apos_entrega'] = (
    DATA_FIM_OBSERVACAO - df_clientes_recompra['data_entrega_referencia']
).dt.total_seconds().div(86400)
# Indicador de auditoria: não muda a definição de recompra.
df_clientes_recompra['recompra_antes_entrega'] = (
    df_clientes_recompra['recomprou']
    & df_clientes_recompra['data_recompra'].lt(df_clientes_recompra['data_entrega_referencia'])
)
assert len(df_clientes_recompra) == len(df_referencia)
assert df_clientes_recompra['customer_unique_id'].is_unique
assert df_clientes_recompra['recomprou'].sum() == len(df_proxima_compra)
assert df_clientes_recompra.loc[
    df_clientes_recompra['recomprou'], 'dias_ate_recompra'
].gt(0).all()
assert df_clientes_recompra['dias_observaveis_apos_entrega'].ge(90).all()


In [ ]:
nota = df_clientes_recompra['avaliacao_referencia']
df_clientes_recompra['grupo_avaliacao'] = 'Sem avaliação'
for grupo, notas in [('Ruim', [1, 2]), ('Neutra', [3]), ('Boa', [4, 5])]:
    df_clientes_recompra.loc[nota.isin(notas), 'grupo_avaliacao'] = grupo

atraso = df_clientes_recompra['atraso_referencia']
df_clientes_recompra['situacao_entrega'] = 'Sem informação'
df_clientes_recompra.loc[atraso.eq(False).fillna(False), 'situacao_entrega'] = 'No prazo'
df_clientes_recompra.loc[atraso.eq(True).fillna(False), 'situacao_entrega'] = 'Atrasado'
df_clientes_recompra['prazo_conhecido'] = atraso.notna()
df_clientes_recompra['referencia_atrasada'] = atraso.eq(True).fillna(False)

df_atraso_por_avaliacao = df_clientes_recompra.groupby(
    'grupo_avaliacao', as_index=False
).agg(
    total_clientes=('customer_unique_id', 'nunique'),
    clientes_com_prazo=('prazo_conhecido', 'sum'),
    clientes_atrasados=('referencia_atrasada', 'sum')
)
df_atraso_por_avaliacao['clientes_sem_prazo'] = (
    df_atraso_por_avaliacao['total_clientes'] - df_atraso_por_avaliacao['clientes_com_prazo']
)
df_atraso_por_avaliacao['pct_atrasados'] = (
    df_atraso_por_avaliacao['clientes_atrasados']
    / df_atraso_por_avaliacao['clientes_com_prazo'].replace(0, float('nan'))
)


def resumir_recompra(dados, grupos):
    resumo = dados.groupby(grupos, as_index=False, observed=True).agg(
        total_clientes=('customer_unique_id', 'nunique'),
        clientes_que_recompraram=('recomprou', 'sum'),
        recompras_antes_entrega=('recompra_antes_entrega', 'sum')
    )
    resumo['clientes_que_nao_recompraram'] = (
        resumo['total_clientes'] - resumo['clientes_que_recompraram']
    )
    resumo['taxa_recompra'] = (
        resumo['clientes_que_recompraram'] / resumo['total_clientes']
    )
    assert resumo['total_clientes'].sum() == len(dados)
    assert resumo['clientes_que_recompraram'].sum() == dados['recomprou'].sum()
    assert resumo['taxa_recompra'].between(0, 1).all()
    return resumo


df_recompra_por_atraso = resumir_recompra(df_clientes_recompra, ['situacao_entrega'])
df_recompra_avaliacao_atraso = resumir_recompra(
    df_clientes_recompra, ['grupo_avaliacao', 'situacao_entrega']
)
assert df_atraso_por_avaliacao['total_clientes'].sum() == len(df_clientes_recompra)
assert df_atraso_por_avaliacao['pct_atrasados'].dropna().between(0, 1).all()


In [ ]:
def formatar_resultado(tabela):
    formatos = {}
    for coluna in tabela.columns:
        if coluna.startswith('pct_') or coluna in ('taxa_recompra', 'taxa_geral_situacao'):
            formatos[coluna] = '{:.2%}'
        elif coluna.endswith('_pp'):
            formatos[coluna] = '{:+.2f} pontos percentuais'
        elif coluna == 'nota_media':
            formatos[coluna] = '{:.2f}'
        elif pd.api.types.is_integer_dtype(tabela[coluna]):
            formatos[coluna] = '{:,.0f}'
    return tabela.style.format(
        formatos, na_rep='Sem informação', decimal=',', thousands='.'
    )

display(formatar_resultado(df_atraso_por_avaliacao))
display(formatar_resultado(df_recompra_por_atraso))
display(formatar_resultado(df_recompra_avaliacao_atraso))


## Gráficos

Montagem dos gráficos da análise.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from io import BytesIO

plt.rcParams['text.parse_math'] = False
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'
graficos_exportar = {}


def registrar_grafico(figura, caminho, dpi=180, bbox_inches='tight'):
    # Captura antes de fechar a figura; grava os arquivos apenas na seção final.
    buffer = BytesIO()
    figura.savefig(buffer, format='png', dpi=dpi, bbox_inches=bbox_inches)
    nome = Path(caminho).name
    if nome in graficos_exportar:
        raise ValueError(f'Nome de gráfico repetido: {nome}')
    graficos_exportar[nome] = buffer.getvalue()


def concluir_grafico(figura, nome, nota):
    figura.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    figura.tight_layout(rect=[0, 0.09, 1, 0.97])
    registrar_grafico(figura, nome)
    plt.show()
    plt.close(figura)


def grafico_situacoes(tabela, grupo, metrica, denominador, titulo, nome, nota):
    # Ausência de grupo permanece NaN: não representa uma taxa observada de zero.
    grupos = tabela[grupo].drop_duplicates().tolist()
    fig, ax = plt.subplots(figsize=(15, max(5, len(grupos) * 0.62 + 2)))
    cores = {'Atrasado': '#C95050', 'No prazo': '#457B9D', 'Sem informação': '#888888'}
    situacoes = [s for s in cores if s in tabela['situacao_entrega'].unique()]
    largura = 0.75 / max(1, len(situacoes))
    for i, situacao in enumerate(situacoes):
        dados = tabela.loc[tabela['situacao_entrega'].eq(situacao)].set_index(grupo).reindex(grupos)
        taxas = dados[metrica].astype(float)
        barras = ax.barh(
            [j + (i - (len(situacoes) - 1) / 2) * largura for j in range(len(grupos))],
            taxas, height=largura, color=cores[situacao], label=situacao
        )
        rotulos = [
            f'{t:.2%}'.replace('.', ',') + f' (n={int(n):,})'.replace(',', '.')
            if pd.notna(t) and pd.notna(n) else ''
            for t, n in zip(taxas, dados[denominador])
        ]
        ax.bar_label(barras, labels=rotulos, padding=3, fontsize=8)
    ax.set_yticks(range(len(grupos)), [str(g).replace('_', ' ') for g in grupos])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.28)
    ax.set_xticks([0, .2, .4, .6, .8, 1])
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel('Proporção')
    ax.set_title(titulo, loc='left', pad=35)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1), ncol=3, frameon=False)
    ax.grid(axis='x', alpha=.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    concluir_grafico(fig, nome, nota)


In [ ]:
dados = df_atraso_por_avaliacao
fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.bar(dados['grupo_avaliacao'], dados['pct_atrasados'].astype(float), color='#C95050')
ax.bar_label(barras, labels=[
    f'{t:.2%}'.replace('.', ',') + f'\n{k:,}/{n:,}'.replace(',', '.')
    if pd.notna(t) else 'Sem prazo'
    for t, k, n in zip(dados['pct_atrasados'], dados['clientes_atrasados'], dados['clientes_com_prazo'])
], padding=4)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title('Atraso na compra de referência por avaliação', loc='left')
ax.spines[['top', 'right']].set_visible(False)
concluir_grafico(fig, '10_atraso_por_avaliacao.png',
    'Denominador: clientes com prazo conhecido em cada grupo de avaliação.\n'
    'Cada cliente aparece uma vez; não representa a distribuição de notas entre atrasados.')

grafico_situacoes(
    df_recompra_avaliacao_atraso, 'grupo_avaliacao', 'taxa_recompra', 'total_clientes',
    'Recompra por avaliação e situação da entrega', '10_recompra_avaliacao_atraso.png',
    'n = clientes da combinação. Recompra pode ocorrer antes da entrega; associação não causal.\n'
    'O grupo sem avaliação é pequeno e tem comportamento distinto.'
)
dados = df_recompra_por_atraso
fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.bar(dados['situacao_entrega'], dados['taxa_recompra'], color='#457B9D')
ax.bar_label(barras, labels=[
    f'{t:.2%}'.replace('.', ',') + f'\n{k:,}/{n:,}'.replace(',', '.')
    for t, k, n in zip(dados['taxa_recompra'], dados['clientes_que_recompraram'], dados['total_clientes'])
], padding=4)
ax.set_ylim(0, max(.01, float(dados['taxa_recompra'].max()) * 1.6))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title('Recompra por situação da entrega de referência', loc='left')
ax.spines[['top', 'right']].set_visible(False)
concluir_grafico(fig, '10_recompra_por_atraso.png',
    'Recompras / clientes elegíveis por situação, incluindo clientes sem avaliação.\n'
    'Mínimo de 90 dias observáveis; recompra não limitada a 90 dias e pode anteceder a entrega.')


## Exportações

Salva as tabelas em Parquet e CSV e os gráficos em PNG.

In [ ]:
PASTA_PARQUET = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'parquet'
PASTA_CSV = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'csv'
for pasta in (PASTA_PARQUET, PASTA_CSV, PASTA_GRAFICOS):
    pasta.mkdir(parents=True, exist_ok=True)

tabelas_exportar = {
    '10_atraso_por_avaliacao': df_atraso_por_avaliacao,
    '10_recompra_por_atraso': df_recompra_por_atraso,
    '10_recompra_avaliacao_atraso': df_recompra_avaliacao_atraso,
}
for nome, original in tabelas_exportar.items():
    tabela = original.reset_index(drop=True).convert_dtypes()
    caminho = PASTA_PARQUET / f'{nome}.parquet'
    tabela.to_parquet(caminho, index=False)
    pd.testing.assert_frame_equal(tabela, pd.read_parquet(caminho), check_exact=True)
    tabela.to_csv(
        PASTA_CSV / f'{nome}.csv', index=False,
        sep=';', decimal=',', encoding='utf-8-sig'
    )
    print(f'Tabela exportada em Parquet e CSV: {nome} ({len(tabela)} linhas)')

for nome, conteudo in graficos_exportar.items():
    (PASTA_GRAFICOS / nome).write_bytes(conteudo)
    print(f'Gráfico exportado: {nome}')
